In this assignment, you will create an optimal portfolio using selection of stocks from the S&P 500 universe. <br>


Your goal is to create a portfolio that maximize the expected sharpe ratio, using your best estimate of stock characteristics prior to the end of 2023 (ie. December 31, 2023). You will then use this optimal weight to backtest the monthly portfolio return in 2024 (assuming monthly rebalancing) and see what the total return of your portoflio is. Check and see if your portfolio outperform a buy-and-hold S&P 500 total return (~25.02%)

You can use the constituents list that we obtained from the first HW (ie. from wikipedia)

Here are some assumptions of the portfolio:
1) The best estimate of the expected return of each constituents is the historial average return
2) You will use the Sharpe Ratio as the objective function
3) You will only long the stock
4) The portfolio should sum to 100%

In [1]:
import datetime
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.optimize import minimize

In [2]:
# get list of S&P500 constituents from wikipedia
wiki_page = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')
symbols = list(wiki_page[0]['Symbol'])

# get historical price data from yahoo finance (auto_adjust default is now True)
stock_data = yf.download(symbols)['Close']

# allow for at least 5 years of training data to get rid of 'bad data' (some stocks don't have data)
cols_to_drop = stock_data.columns[stock_data[stock_data.index.year>=2019].isna().any()]
stock_data.drop(columns=cols_to_drop, inplace=True)

# get monthly returns and split into train/test
monthly_returns = stock_data.resample("ME").last().dropna().pct_change()
monthly_returns_train = monthly_returns[monthly_returns.index.year<=2023]
monthly_returns_test = monthly_returns[monthly_returns.index.year==2024]

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  503 of 503 completed

2 Failed downloads:
['BRK.B']: YFTzMissingError('possibly delisted; no timezone found')
['BF.B']: YFPricesMissingError('possibly delisted; no price data found  (1d 1926-04-11 -> 2025-03-17)')


In [3]:
# maximize sharpe ratio to find optimal portfolio weights

# objective function (negative sharp since optimization has to be done on minimization)
def negative_sharpe(weights, returns, risk_free_rt=0.04):
    portfolio_return = np.dot(weights, returns)
    portfolio_volatility = np.sqrt(np.dot(np.dot(weights.T, np.cov(returns.T)), weights))
    sharpe_ratio = (portfolio_return - risk_free_rt) / portfolio_volatility
    return -sharpe_ratio

# perform optimization
num_stocks = monthly_returns_train.shape[1]
mean_monthly_returns_train = monthly_returns_train.mean()
initial_weights = np.ones(num_stocks) / num_stocks  # initial guess of the weights
bounds = [(0,1) for i in range(num_stocks)]  # no short selling
constraints = ({'type':'eq', 'fun': lambda W: np.sum(W) - 1 })  # weights must sum to 100%
result = minimize(negative_sharpe, initial_weights, (mean_monthly_returns_train), method='SLSQP',
                  bounds=bounds, constraints=constraints)

result

 message: Optimization terminated successfully
 success: True
  status: 0
     fun: -6.181481286247218
       x: [ 3.257e-13  3.568e-12 ...  4.117e-13  5.877e-13]
     nit: 18
     jac: [-3.943e+00 -7.753e+00 ... -4.041e+00 -4.255e+00]
    nfev: 8746
    njev: 18

In [4]:
# check that the weights sum up to 1
result.x.sum()

np.float64(1.0000000002744573)

In [5]:
# backtest 2024 using optimial weights found above

# initialize portfolio value
portfolio_values = [1]

optimal_weights = result.x

# run backtest
for i in range(len(monthly_returns_test)):
    monthly_return = monthly_returns_test.iloc[i]
    portfolio_return = np.dot(optimal_weights, monthly_return)
    new_portfolio_value = portfolio_values[-1] * (1 + portfolio_return)
    portfolio_values.append(new_portfolio_value)

portfolio_values

[1,
 np.float64(1.07592369150504),
 np.float64(1.2951846467487378),
 np.float64(1.3539802695480891),
 np.float64(1.2621635129232125),
 np.float64(1.3589270077641644),
 np.float64(1.278984435164772),
 np.float64(1.354209399486018),
 np.float64(1.2224370484278182),
 np.float64(1.220161809239411),
 np.float64(1.0322921416737507),
 np.float64(1.054353448491892),
 np.float64(1.0194427332509322)]

In [6]:
print('Sharpe-optimized portfolio performance in 2024:')
print(round(float(portfolio_values[-1] / 1 - 1)*100, 2), '%')

Sharpe-optimized portfolio performance in 2024:
1.94 %


Shown above, the portfolio earns only **1.9%** in 2024. This significantly **underperforms** the buy-and-hold S&P 500 total return (~25.02%) of that same time period.